# 03_gnn_text_v2 — SOTA Traffic GNN

Thin notebook: all components live in `src/model_v2.py`, `src/train.py`.

Upgrades over v1:
- **GatedTCN** — gated dilated convolutions (fixed truncation)
- **DiffusionGCN** — K-hop spatial aggregation (fixed extra hop)
- **AdaptiveAdj** — learned adjacency mixed with static road network
- **CrossModalFusion** — multi-head cross-attention over text tokens
- **Qwen3.5-0.8B** — token-level LLM text encoder (frozen + LoRA, fixed speed)
- **Gradient clipping + LR scheduler** — stable training
- **Auto-caching** — embeddings recompute only when texts change

## Section 1: Setup & Data

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from src import (
    load_train_speeds, load_train_texts,
    load_adjacency,
    build_windows, compute_norm_stats, normalize, denormalize,
    train_val_split, compute_mse,
    cache_embeddings,
)
from src.model_v2 import SOTATrafficGNN
from src.train import (
    build_adj_tensor,
    encode_texts_minilm, encode_texts_qwen, load_qwen,
    train_one_config,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
s1, s2 = load_train_speeds()
t1, t2 = load_train_texts()
adj_raw = load_adjacency()

X1, T1, Y1 = build_windows(s1, t1)
X2, T2, Y2 = build_windows(s2, t2)

X_all = np.concatenate([X1, X2], axis=0).astype(np.float32)
Y_all = np.concatenate([Y1, Y2], axis=0).astype(np.float32)
T_all = np.array(T1 + T2)

mean, std = compute_norm_stats(np.concatenate([s1, s2], axis=0))
X_norm = normalize(X_all, mean, std)
Y_norm = normalize(Y_all, mean, std)

X_tr, _, Y_tr, X_va, _, Y_va = train_val_split(X_norm, None, Y_norm, val_frac=0.2)
T_tr = T_all[:len(X_tr)]
T_va = T_all[len(X_tr):]

adj_t = build_adj_tensor(adj_raw).to(DEVICE)

print(f"Train: {X_tr.shape}, Val: {X_va.shape}")
print(f"Adj: {adj_t.shape}, nnz={(adj_t > 0).sum().item()}")

## Section 2: Text Encoders (auto-cached)

MiniLM (384-dim) and Qwen3.5-0.8B (1024-dim token-level). Both cached with content-hash — recompute only when train/val texts change.

In [ ]:
T_emb_tr_minilm, T_emb_va_minilm = cache_embeddings(
    T_tr.tolist(), T_va.tolist(),
    encoder_fn=encode_texts_minilm,
    cache_name="minilm",
    encoder_kwargs={"batch_size": 256},
)
print(f"MiniLM: train={T_emb_tr_minilm.shape}, val={T_emb_va_minilm.shape}")

In [ ]:
qwen_model, qwen_tokenizer = load_qwen("Qwen/Qwen3.5-0.8B", device=DEVICE)

def encode_qwen_wrapper(texts, batch_size=32):
    return encode_texts_qwen(texts, qwen_model, qwen_tokenizer, DEVICE,
                             batch_size=batch_size)

T_emb_tr_qwen, T_emb_va_qwen = cache_embeddings(
    T_tr.tolist(), T_va.tolist(),
    encoder_fn=encode_qwen_wrapper,
    cache_name="qwen",
    encoder_kwargs={"batch_size": 32},
)
print(f"Qwen3.5: train={T_emb_tr_qwen.shape}, val={T_emb_va_qwen.shape}")

In [ ]:
class TrafficDS(Dataset):
    def __init__(self, X, T_emb, Y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.T = torch.tensor(T_emb, dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.T[i], self.Y[i]


def make_loaders(T_emb_tr, T_emb_va, batch_size=32):
    tr = TrafficDS(X_tr, T_emb_tr, Y_tr)
    va = TrafficDS(X_va, T_emb_va, Y_va)
    return (
        DataLoader(tr, batch_size=batch_size, shuffle=True, pin_memory=True),
        DataLoader(va, batch_size=batch_size, shuffle=False, pin_memory=True),
    )

print("DataLoaders ready.")

## Section 3: Smoke Test

Verify all model variants instantiate and forward-pass correctly.

In [ ]:
x_dummy = torch.randn(2, 15, 1260)
adj_dummy = torch.eye(1260) * 0.5

for name, text_dim, text_shape in [
    ("MiniLM concat", 384, (2, 384)),
    ("MiniLM cross", 384, (2, 384)),
    ("Qwen cross", 1024, (2, 32, 1024)),
    ("Speed-only", 384, (2, 384)),
]:
    use_text = name != "Speed-only"
    fusion = "cross" if "cross" in name else "concat"
    model = SOTATrafficGNN(
        d_model=64, num_tcn_layers=4, num_gcn_layers=4,
        num_hops=2, text_dim=text_dim, n_heads=4, dropout=0.2,
        use_text=use_text, fusion=fusion, use_adaptive_adj=True,
    )
    t = torch.randn(text_shape)
    with torch.no_grad():
        out = model(x_dummy, t, adj_dummy)
    n = sum(p.numel() for p in model.parameters())
    print(f"  {name:<20} params={n:>8,}  out={list(out.shape)}")

print("All variants OK.")

## Section 4: Ablation Table

Train each config and compare validation MSE.

| # | Name | Temporal | Spatial | Adjacency | Fusion | Text |
|---|---|---|---|---|---|---|
| A | Baseline (v1) | Conv1D+pool | 1-hop GCN | Static | Concat | MiniLM |
| B | + Gated TCN | GatedTCN | 1-hop GCN | Static | Concat | MiniLM |
| C | + Diffusion | GatedTCN | K-hop (K=2) | Static | Concat | MiniLM |
| D | + Adaptive | GatedTCN | K-hop (K=2) | Learned | Concat | MiniLM |
| E | + Cross-modal | GatedTCN | K-hop (K=2) | Learned | Cross-attn | MiniLM |
| F | + Qwen3.5 | GatedTCN | K-hop (K=2) | Learned | Cross-attn | Qwen3.5 |

In [ ]:
@torch.no_grad()
def eval_mse(model, loader, adj, device):
    model.eval()
    preds, targets = [], []
    for Xb, Tb, Yb in loader:
        Xb, Tb, Yb = Xb.to(device), Tb.to(device), Yb.to(device)
        pb = model(Xb, Tb, adj)
        preds.append(pb.cpu().numpy())
        targets.append(Yb.cpu().numpy())
    yp = denormalize(np.concatenate(preds, axis=0), mean, std)
    yt = denormalize(np.concatenate(targets, axis=0), mean, std)
    return compute_mse(yp, yt)


def run_ablation(name, tcn_layers, gcn_layers, num_hops,
                 use_adaptive_adj, fusion, text_dim,
                 T_emb_tr, T_emb_va, batch_size=32):
    print(f"{name}...")
    model = SOTATrafficGNN(
        d_model=64, num_tcn_layers=tcn_layers, num_gcn_layers=gcn_layers,
        num_hops=num_hops, text_dim=text_dim, n_heads=4, dropout=0.2,
        use_text=True, fusion=fusion, use_adaptive_adj=use_adaptive_adj,
    ).to(DEVICE)
    tl, vl = make_loaders(T_emb_tr, T_emb_va, batch_size=batch_size)
    _, n, ep = train_one_config(model, tl, vl, adj_t, DEVICE,
                                 epochs=50, patience=10)
    mse = eval_mse(model, vl, adj_t, DEVICE)
    print(f"  -> MSE={mse:.4f}  params={n:,}  epochs={ep}")
    return mse

print("Ablation helpers ready.")

In [ ]:
results = {}

results["A: Baseline (v1)"] = run_ablation(
    "A", tcn_layers=0, gcn_layers=4, num_hops=1,
    use_adaptive_adj=False, fusion="concat", text_dim=384,
    T_emb_tr=T_emb_tr_minilm, T_emb_va=T_emb_va_minilm,
)

results["B: + GatedTCN"] = run_ablation(
    "B", tcn_layers=4, gcn_layers=4, num_hops=1,
    use_adaptive_adj=False, fusion="concat", text_dim=384,
    T_emb_tr=T_emb_tr_minilm, T_emb_va=T_emb_va_minilm,
)

results["C: + Diffusion K=2"] = run_ablation(
    "C", tcn_layers=4, gcn_layers=4, num_hops=2,
    use_adaptive_adj=False, fusion="concat", text_dim=384,
    T_emb_tr=T_emb_tr_minilm, T_emb_va=T_emb_va_minilm,
)

results["D: + AdaptiveAdj"] = run_ablation(
    "D", tcn_layers=4, gcn_layers=4, num_hops=2,
    use_adaptive_adj=True, fusion="concat", text_dim=384,
    T_emb_tr=T_emb_tr_minilm, T_emb_va=T_emb_va_minilm,
)

results["E: + CrossModal"] = run_ablation(
    "E", tcn_layers=4, gcn_layers=4, num_hops=2,
    use_adaptive_adj=True, fusion="cross", text_dim=384,
    T_emb_tr=T_emb_tr_minilm, T_emb_va=T_emb_va_minilm,
)

results["F: + Qwen3.5"] = run_ablation(
    "F", tcn_layers=4, gcn_layers=4, num_hops=2,
    use_adaptive_adj=True, fusion="cross", text_dim=1024,
    T_emb_tr=T_emb_tr_qwen, T_emb_va=T_emb_va_qwen, batch_size=16,
)

## Summary

In [ ]:
print("=" * 64)
print(f"{'Config':<30} {'Val MSE':>10}")
print("-" * 42)
for name, mse in sorted(results.items(), key=lambda x: x[1]):
    print(f"{name:<30} {mse:>10.4f}")
best = min(results, key=results.get)
print(f"\nBest: {best} ({results[best]:.4f})")